In [1]:
import os
print(os.getcwd())

/home/sherif/Desktop/credit-lens/notebooks


In [2]:
import pandas as pd

demographics = pd.read_csv("../data/traindemographics.csv")
performance = pd.read_csv("../data/trainperf.csv")
prevloans = pd.read_csv("../data/trainprevloans.csv")

print(demographics.shape)
print(performance.shape)
print(prevloans.shape)


(4346, 9)
(4368, 10)
(18183, 12)


In [3]:
demographics.info()
print("---")
performance.info()
print("---")
prevloans.info()

<class 'pandas.DataFrame'>
RangeIndex: 4346 entries, 0 to 4345
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  4346 non-null   str    
 1   birthdate                   4346 non-null   str    
 2   bank_account_type           4346 non-null   str    
 3   longitude_gps               4346 non-null   float64
 4   latitude_gps                4346 non-null   float64
 5   bank_name_clients           4346 non-null   str    
 6   bank_branch_clients         51 non-null     str    
 7   employment_status_clients   3698 non-null   str    
 8   level_of_education_clients  587 non-null    str    
dtypes: float64(2), str(7)
memory usage: 305.7 KB
---
<class 'pandas.DataFrame'>
RangeIndex: 4368 entries, 0 to 4367
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customerid     4368 non-null   str 

In [4]:
performance["good_bad_flag"].value_counts()
performance["good_bad_flag"].value_counts(normalize=True)

good_bad_flag
Good    0.782051
Bad     0.217949
Name: proportion, dtype: float64

In [5]:
df = performance.merge(demographics, on="customerid", how="left")
print(df.shape)
df.head()

(4376, 18)


,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,1972-01-15 00:00:00.000000,Other,3.432010,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good,1985-08-23 00:00:00.000000,Savings,3.885298,7.320700,GT Bank,"DUGBE,IBADAN",Permanent,Graduate
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good,1984-09-18 00:00:00.000000,Other,11.139350,10.292041,EcoBank,NaN,Permanent,NaN
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good,1977-10-10 00:00:00.000000,Savings,3.985770,7.491708,First Bank,NaN,Permanent,NaN
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good,1986-09-07 00:00:00.000000,Other,7.457913,9.076574,GT Bank,NaN,Permanent,Primary


In [6]:
prevloans["approveddate"] = pd.to_datetime(prevloans["approveddate"])
prevloans["closeddate"] = pd.to_datetime(prevloans["closeddate"])
prevloans["firstduedate"] = pd.to_datetime(prevloans["firstduedate"])
prevloans["firstrepaiddate"] = pd.to_datetime(prevloans["firstrepaiddate"])

prevloans["days_early_late"] = (prevloans["firstrepaiddate"] - prevloans["firstduedate"]).dt.days

prev_agg = prevloans.groupby("customerid").agg(
    prev_loan_count=("systemloanid", "count"),
    avg_days_early_late=("days_early_late", "mean"),
    max_days_late=("days_early_late", "max"),
    avg_loanamount=("loanamount", "mean"),
).reset_index()

prev_agg.head()

,customerid,prev_loan_count,avg_days_early_late,max_days_late,avg_loanamount
0,8a1088a0484472eb01484669e3ce4e0b,1,6.000000,6,10000.000000
1,8a1a1e7e4f707f8b014f797718316cad,4,-0.250000,1,17500.000000
2,8a1a32fc49b632520149c3b8fdf85139,7,-0.428571,1,12857.142857
3,8a1eb5ba49a682300149c3c068b806c7,8,-3.125000,8,16250.000000
4,8a1edbf14734127f0147356fdb1b1eb2,2,-4.000000,0,10000.000000


In [7]:
df = df.merge(prev_agg, on="customerid", how="left")

df["prev_loan_count"] = df["prev_loan_count"].fillna(0)
df["employment_status_clients"] = df["employment_status_clients"].fillna("Unknown")
df["was_referred"] = df["referredby"].notna().astype(int)

df = df.drop(columns=["bank_branch_clients", "level_of_education_clients", "referredby"])

print(df.shape)
df.isnull().sum()

(4376, 20)


customerid                      0
systemloanid                    0
loannumber                      0
approveddate                    0
creationdate                    0
loanamount                      0
totaldue                        0
termdays                        0
good_bad_flag                   0
birthdate                    1099
bank_account_type            1099
longitude_gps                1099
latitude_gps                 1099
bank_name_clients            1099
employment_status_clients       0
prev_loan_count                 0
avg_days_early_late             9
max_days_late                   9
avg_loanamount                  9
was_referred                    0
dtype: int64

In [8]:
print(demographics["customerid"].duplicated().sum())
print(prev_agg["customerid"].duplicated().sum())

12
0


In [9]:
perf_ids = set(performance["customerid"])
demo_ids = set(demographics["customerid"])
print(len(perf_ids))
print(len(demo_ids))
print(len(perf_ids & demo_ids))
print(len(perf_ids - demo_ids))

4368
4334
3269
1099


In [10]:
demographics = demographics.drop_duplicates(subset="customerid", keep="first")

df = performance.merge(demographics, on="customerid", how="left")
df = df.merge(prev_agg, on="customerid", how="left")

print(df.shape)

(4368, 22)


In [11]:
df["prev_loan_count"] = df["prev_loan_count"].fillna(0)
df["employment_status_clients"] = df["employment_status_clients"].fillna("Unknown")
df["was_referred"] = df["referredby"].notna().astype(int)

df = df.drop(columns=["bank_branch_clients", "level_of_education_clients", "referredby"])

print(df.shape)
df.isnull().sum()

(4368, 20)


customerid                      0
systemloanid                    0
loannumber                      0
approveddate                    0
creationdate                    0
loanamount                      0
totaldue                        0
termdays                        0
good_bad_flag                   0
birthdate                    1099
bank_account_type            1099
longitude_gps                1099
latitude_gps                 1099
bank_name_clients            1099
employment_status_clients       0
prev_loan_count                 0
avg_days_early_late             9
max_days_late                   9
avg_loanamount                  9
was_referred                    0
dtype: int64

In [12]:
df["has_demographics"] = df["birthdate"].notna().astype(int)

df["bank_account_type"] = df["bank_account_type"].fillna("Unknown")
df["bank_name_clients"] = df["bank_name_clients"].fillna("Unknown")

df["longitude_gps"] = df["longitude_gps"].fillna(df["longitude_gps"].median())
df["latitude_gps"] = df["latitude_gps"].fillna(df["latitude_gps"].median())

df["avg_days_early_late"] = df["avg_days_early_late"].fillna(0)
df["max_days_late"] = df["max_days_late"].fillna(0)
df["avg_loanamount"] = df["avg_loanamount"].fillna(0)

df = df.drop(columns=["birthdate"])

print(df.shape)
df.isnull().sum()

(4368, 20)


customerid                   0
systemloanid                 0
loannumber                   0
approveddate                 0
creationdate                 0
loanamount                   0
totaldue                     0
termdays                     0
good_bad_flag                0
bank_account_type            0
longitude_gps                0
latitude_gps                 0
bank_name_clients            0
employment_status_clients    0
prev_loan_count              0
avg_days_early_late          0
max_days_late                0
avg_loanamount               0
was_referred                 0
has_demographics             0
dtype: int64